# Manejo o Administracion de errores y excepciones en Python

Algo indispensable es aprender : Patrones de Diseño

## Flujo General de una Excepcion

1. Se ejecuta codigo dentro de un bloque `try`
2. Ocurre un error o se usa explicitamente el comando `raise`
3. Python crea o instancia un objeto de una excepcion
4. Se busca un bloque `except` comptatible
    * Primero en el mismo bloque
    * Luego en funciones superiores (propagacion)
    * Si no se captura, el programa debe terminar en `traceback`
5. Si no ocurre excepcion -> Se ejecuta el bloque `else`
6. Independientemente del resultado -> Se ejecuta el bloque `finally`

# El comando `raise`

* Lanza una excepcion explicitamente

In [ ]:
import math

# 1) Relanzar la excepcion actual
try:
    math.sqrt(-1)
except ValueError:
    print("Error de operacion")  # aqui iria logging
    try:
        raise  # relanza la misma excepcion
    except ValueError as e:
        print("Relanzada:", e)

# 2) Encadenamiento explicito (from)
try:
    int("Cuarenta y cuatro")
except ValueError as e:
    try:
        raise TypeError("Datos no convertibles a entero") from e
    except TypeError as e2:
        print(e2, "<- causa:", repr(e2.__cause__))

# Mal uso comun: try/except desnudo para un valor por defecto
diccionario = {}
valor = diccionario.get("clave", "valor_por_defecto")  # mejor que try/except

# La palabra reservada `assert`

* Verifica una condicion, luego lanza una excepcion automaticamente (AssertionError) si la condicion es falsa


# Jerarquia de Excepciones

* BaseException
    * Exception
        * TypeError
        * ValueError
        * KeyError
        * IOError
        * ZeroDivisionError
        * ...
        * **NUestras excepciones personalizadas**

* SystemExit
* KeyboardInterrupt

# Definicion de Excepciones Personalizadas

1. Heredar siempre de `Exception` (No de `BaseException`)
2. Utilizar el sufijo `Error` (Por convencion)
3. Definir una jerarquia base para nuestro dominio de aplicacion o modulo.
4. Añadir atributos y mensajes informativos
    - Definir `__init__` para recibir informacion contextual y almacenarla como atributos
    - Sobreescribir : `__str__`
5. Mantener la excepcion ligera (evitar logica compleja)
6. Documentar las excepciones personalizadas
7. Usar herencia multiple solo si es necesario (con cuidado)
8. No abusar de las excepciones personalizadas para casos cubiertos por el estandar (modulo : `builtins`)
9. Lanzar siempre con `raise` y proporcionar contexto con `from`
10. Usar clases vacias (solo `pass`) cuando solo importe el tipo
11. Considerar usar `__slots__` para ahorrar memoria (opcional)
12. Exportar las excepciones en `__init__` del modulo

## Jerarquia propia de Excepciones

In [ ]:
# Jerarquia propia de excepciones (docstrings abreviados)
class ValidationStartupError(Exception): """Error base de validacion."""
class BlockChainFormatError(ValidationStartupError): """Formato de libro contable."""
class RangeIDError(BlockChainFormatError): """ID fuera de rango."""

class DBError(Exception): """Error general de base de datos."""
class ConexionError(DBError): """Error al conectar."""
class ConsultaError(DBError): """Error al ejecutar consulta SQL."""

def conectar_db():
    raise ConexionError("Fallo al conectar : Servidor no disponible")

def ejecutar_consulta():
    raise ConsultaError("Consulta invalida : Tabla Inexistente")

# Capturar por la clase base cubre toda la jerarquia
for f in (conectar_db, ejecutar_consulta):
    try:
        f()
    except DBError as e:
        print(f"Rutina de mantenimiento -> {type(e).__name__}: {e}")

In [ ]:
# Excepciones de validacion + funciones
class EdadInvalidaError(Exception): """Edad < 0 o >= 115 (regla del area usuaria)."""
class EmailInvalidoError(Exception): """Email sin '@' o '.'"""
class CalificacionInvalidaError(Exception): """Calificacion fuera de [0,20]."""
class CoordenadaInvalidaError(Exception): """Latitud/longitud fuera de rango."""

def validar_edad(edad):
    if not 0 <= edad < 115:
        raise EdadInvalidaError(f"Edad invalida : {edad} : debe estar en [0,115)")
    return f"Edad valida : {edad}"

def validar_email(email):
    if "@" not in email or "." not in email:
        raise EmailInvalidoError(f"Email invalido : {email}")
    return "Email valido"

def registrar_calificacion(calif):
    if not isinstance(calif, (int, float)):
        raise TypeError("La calificacion debe ser numerica")
    if not 0 <= calif <= 20:
        raise CalificacionInvalidaError(f"Calificacion invalida : {calif}")
    return f"Calificacion registrada : {calif}"

def validar_coordenada(coord):
    lat, lon = coord
    if not -90 <= lat <= 90:
        raise CoordenadaInvalidaError(f"Latitud invalida : {lat}")
    if not -180 <= lon <= 180:
        raise CoordenadaInvalidaError(f"Longitud invalida : {lon}")
    return "Coordenadas validas"

def probar(funcion, datos, errores):
    """Ejecuta funcion sobre cada dato; cuenta y muestra los errores capturados."""
    fallos = 0
    for d in datos:
        try:
            print(funcion(d))
        except errores as e:
            fallos += 1
            print(f"Error capturado : {e}")
    print(f"Errores: {fallos}/{len(datos)}\n")

probar(validar_edad, [0, 25, -2, -36, 45, 98, 116], EdadInvalidaError)
probar(validar_email, ["azamudio@pit788.uni.edu.pe", "usuario_arroba_linux.pe",
                       "123456789_mit.edu.pe", "wcastillo@uni.pe"], EmailInvalidoError)
probar(registrar_calificacion, ["veinte", 1, -13, 0, "cinco", "@"],
       (CalificacionInvalidaError, TypeError))
probar(validar_coordenada, [(-407.128, -74.0060), (100, -47), (-11, 19)],
       CoordenadaInvalidaError)

# Paradigma de la Programacion Orientada a Objetos

Es un paradigma que organiza el software en objetos, los cuales son instancias de clases

In [ ]:
a, b = 12, int("12")
print(type(a), type(b), a == b)

# help(int), help(BaseException), help(list)  -> documentacion de clases constructoras

class list_game(list):
    """Clase hija de list."""

juego1 = list_game()
print(type(juego1), isinstance(juego1, list))

Aprender a manejar:
- Herencia
- Encapsulamiento
- Polimorfismo
- Abstraccion

## Referencias Bilbiograficas

* Meyer, B. (1997). Object-oriented software construction (Vol. 2, pp. 331-410). Englewood Cliffs: Prentice hall.

* Webster, B. F. (1995). Pitfalls of object-oriented development. M & T Books.

* Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1995). Design patterns (Vol. 2). Reading: Addison wesley.

* Page-Jones, M. (2000). Fundamentals of object-oriented design in UML. Pearson Education India.

# ¿Que es una clase en Python?

Una clase es un objetoo de primer orden que define una plantilla (blueprint) para crear instancias conocidas como objetos

|Programacion Estructurada | programacion orientada a objetos |
|---------------------------|------------------------|
|Funcion|Objeto/Clase|
| Variables globales o pasadas por referencia | Encapsulado dentro del objeto|
| Secuencial, Explicito, Jerarquico| Basado en mensajes entre objetos |
| Las funciones dependen de estructuras de datos compartidas | Los objetos depende de interfaces/contratos mas no de implementaciones|
|Agrupacion por proximidad funcional | Agrupacion por responsabilidad unica|
